In [6]:
import requests, time, pandas as pd

In [7]:
API_KEY = "60b668449ffd990d9e90f90a2603ba96"

def fetch_movie(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    r = requests.get(url, params={"api_key": API_KEY})
    if r.status_code != 200:
        return None
    d = r.json()
    # Extract only what we need and flatten it
    return {
        "tmdb_id":         d.get("id"),
        "title":           d.get("title"),
        "runtime":         d.get("runtime"),           # can be None
        "language":        d.get("original_language"),
        "studio":          d["production_companies"][0]["name"] 
                           if d.get("production_companies") else None,
        "studio_country":  d["production_companies"][0].get("origin_country")
                           if d.get("production_companies") else None,
        "release_date":    d.get("release_date"),
        "status":          d.get("status"),            # Released, Rumored, etc.
    }

In [8]:
kaggle_df = pd.read_csv("tmdb_5000_movies.csv")
ids = kaggle_df["id"].dropna().astype(int).tolist()[:300]  # limit for demo

rows = []
for mid in ids:
    row = fetch_movie(mid)
    if row:
        rows.append(row)
    time.sleep(0.25)  # TMDB allows ~4 requests/second on free tier

api_df = pd.DataFrame(rows)

In [9]:
print (api_df)

     tmdb_id                                     title  runtime language  \
0      19995                                    Avatar      162       en   
1        285  Pirates of the Caribbean: At World's End      169       en   
2     206647                                   Spectre      148       en   
3      49026                     The Dark Knight Rises      165       en   
4      49529                               John Carter      132       en   
..       ...                                       ...      ...      ...   
295    37710                               The Tourist      103       en   
296     9946                               End of Days      122       en   
297     1372                             Blood Diamond      143       en   
298   106646                   The Wolf of Wall Street      180       en   
299      414                            Batman Forever      121       en   

                      studio studio_country release_date    status  
0         Dune Ent

In [10]:
api_df.dropna(subset=["tmdb_id", "title"], inplace=True)
api_df["runtime"] = pd.to_numeric(api_df["runtime"], errors="coerce")
api_df = api_df[api_df["runtime"] > 40]          # remove shorts/errors
api_df["language"] = api_df["language"].str.upper()
api_df["release_date"] = pd.to_datetime(api_df["release_date"], errors="coerce")
api_df.drop_duplicates(subset=["tmdb_id"], inplace=True)

In [11]:
print(api_df.head())

   tmdb_id                                     title  runtime language  \
0    19995                                    Avatar      162       EN   
1      285  Pirates of the Caribbean: At World's End      169       EN   
2   206647                                   Spectre      148       EN   
3    49026                     The Dark Knight Rises      165       EN   
4    49529                               John Carter      132       EN   

                    studio studio_country release_date    status  
0       Dune Entertainment             US   2009-12-16  Released  
1  Jerry Bruckheimer Films             US   2007-05-19  Released  
2      Metro-Goldwyn-Mayer             US   2015-10-26  Released  
3                  Syncopy             GB   2012-07-17  Released  
4     Walt Disney Pictures             US   2012-03-07  Released  


In [13]:
print (api_df)

     tmdb_id                                     title  runtime language  \
0      19995                                    Avatar      162       EN   
1        285  Pirates of the Caribbean: At World's End      169       EN   
2     206647                                   Spectre      148       EN   
3      49026                     The Dark Knight Rises      165       EN   
4      49529                               John Carter      132       EN   
..       ...                                       ...      ...      ...   
295    37710                               The Tourist      103       EN   
296     9946                               End of Days      122       EN   
297     1372                             Blood Diamond      143       EN   
298   106646                   The Wolf of Wall Street      180       EN   
299      414                            Batman Forever      121       EN   

                      studio studio_country release_date    status  
0         Dune Ent

In [14]:
api_df.to_csv("api_cleaned.csv", index=False)
print(api_df.head())

   tmdb_id                                     title  runtime language  \
0    19995                                    Avatar      162       EN   
1      285  Pirates of the Caribbean: At World's End      169       EN   
2   206647                                   Spectre      148       EN   
3    49026                     The Dark Knight Rises      165       EN   
4    49529                               John Carter      132       EN   

                    studio studio_country release_date    status  
0       Dune Entertainment             US   2009-12-16  Released  
1  Jerry Bruckheimer Films             US   2007-05-19  Released  
2      Metro-Goldwyn-Mayer             US   2015-10-26  Released  
3                  Syncopy             GB   2012-07-17  Released  
4     Walt Disney Pictures             US   2012-03-07  Released  
